# 11.28 — Imitation & inverse RL

Imitation asks what expert actions reveal about behavior and intent.

Behavior cloning learns action probabilities from demonstrations, while inverse RL estimates rewards that make expert behavior valuable. The notebook compares both on the same ladder. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1125
rng = np.random.default_rng(SEED)
ACTIONS = np.array([0, 1, 2, 3])
DELTAS = {
    0: np.array([-1, 0]),
    1: np.array([0, 1]),
    2: np.array([1, 0]),
    3: np.array([0, -1]),
}
ACTION_NAMES = np.array(["up", "right", "down", "left"])


def discounted_return(rewards, gamma):
    total = 0.0
    for t, reward in enumerate(rewards):
        total += (gamma ** t) * reward
    return float(total)


def td_update(q_old, reward, next_value, alpha, gamma):
    target = reward + gamma * next_value
    q_new = q_old + alpha * (target - q_old)
    return float(target), float(q_new)


def make_grid_env(name, width, height, start, goals, walls=None, slip=0.0, wind=None, step_cost=-0.02, max_steps=40):
    walls = set(walls or [])
    goals = dict(goals)
    wind = wind or {}
    states = []
    index = {}
    for row in range(height):
        for col in range(width):
            pos = (row, col)
            if pos not in walls:
                index[pos] = len(states)
                states.append(pos)
    env = {
        "name": name,
        "width": width,
        "height": height,
        "start": start,
        "goals": goals,
        "walls": walls,
        "slip": slip,
        "wind": wind,
        "step_cost": step_cost,
        "max_steps": max_steps,
        "states": states,
        "index": index,
    }
    return env


def move_position(env, pos, action):
    drift = env["wind"].get(pos, np.array([0, 0]))
    raw = np.array(pos) + DELTAS[int(action)] + drift
    row = int(np.clip(raw[0], 0, env["height"] - 1))
    col = int(np.clip(raw[1], 0, env["width"] - 1))
    candidate = (row, col)
    if candidate in env["walls"]:
        return pos
    return candidate


def sample_step(env, pos, action, local_rng):
    chosen = int(action)
    if local_rng.random() < env["slip"]:
        chosen = int(local_rng.choice(ACTIONS))
    next_pos = move_position(env, pos, chosen)
    reward = env["step_cost"]
    done = False
    if next_pos in env["goals"]:
        reward += env["goals"][next_pos]
        done = True
    return next_pos, float(reward), done


def greedy_action(q_values, state_index, allowed=None):
    values = q_values[state_index].copy()
    if allowed is not None:
        values = np.where(allowed[state_index], values, -1e9)
    return int(np.argmax(values))


def rollout_return(env, q_values, episodes=20, gamma=0.9, support=None, seed=0, goal=None):
    local_rng = np.random.default_rng(seed)
    returns = []
    successes = []
    for episode in range(episodes):
        pos = env["start"]
        total = 0.0
        discount = 1.0
        success = 0.0
        for step in range(env["max_steps"]):
            state = env["index"][pos]
            action = greedy_action(q_values, state, support)
            pos, reward, done = sample_step(env, pos, action, local_rng)
            if goal is not None:
                reward = 1.0 if pos == goal else -0.02
                done = pos == goal
            total += discount * reward
            discount *= gamma
            if done:
                success = 1.0
                break
        returns.append(total)
        successes.append(success)
    return float(np.mean(returns)), float(np.mean(successes))


def q_learning(env, episodes=120, alpha=0.35, gamma=0.9, epsilon=0.2, shaping=None, goal=None, seed=0):
    local_rng = np.random.default_rng(seed)
    q_values = np.zeros((len(env["states"]), len(ACTIONS)))
    history = []
    for episode in range(episodes):
        pos = env["start"]
        total = 0.0
        discount = 1.0
        for step in range(env["max_steps"]):
            state = env["index"][pos]
            if local_rng.random() < epsilon:
                action = int(local_rng.choice(ACTIONS))
            else:
                action = int(np.argmax(q_values[state]))
            next_pos, reward, done = sample_step(env, pos, action, local_rng)
            if goal is not None:
                reward = 1.0 if next_pos == goal else -0.02
                done = next_pos == goal
            shaped_reward = reward
            if shaping is not None:
                shaped_reward += shaping(pos, next_pos)
            next_state = env["index"][next_pos]
            target = shaped_reward + gamma * np.max(q_values[next_state]) * (1.0 - float(done))
            q_values[state, action] += alpha * (target - q_values[state, action])
            total += discount * reward
            discount *= gamma
            pos = next_pos
            if done:
                break
        history.append(total)
    return q_values, np.array(history)


def value_iteration(env, gamma=0.9, reward_bonus=None, iterations=80):
    values = np.zeros(len(env["states"]))
    q_values = np.zeros((len(env["states"]), len(ACTIONS)))
    for iteration in range(iterations):
        new_values = values.copy()
        for pos in env["states"]:
            state = env["index"][pos]
            candidates = []
            for action in ACTIONS:
                next_pos = move_position(env, pos, int(action))
                reward = env["step_cost"] + env["goals"].get(next_pos, 0.0)
                if reward_bonus is not None:
                    reward += reward_bonus(pos, next_pos)
                done = next_pos in env["goals"]
                next_state = env["index"][next_pos]
                candidates.append(reward + gamma * values[next_state] * (1.0 - float(done)))
            q_values[state] = np.array(candidates)
            new_values[state] = float(np.max(candidates))
        values = new_values
    return values, q_values


def potential_to_goal(env, goal):
    distances = {}
    for pos in env["states"]:
        distances[pos] = abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])
    max_distance = max(distances.values()) + 1.0
    return {pos: -dist / max_distance for pos, dist in distances.items()}


def rl_ladder(goal_conditioned=False):
    rungs = []
    rungs.append(make_grid_env("D1 2-state chain", 2, 1, (0, 0), {(0, 1): 1.0}, max_steps=4))
    rungs.append(make_grid_env("D2 slippery 3-state chain", 3, 1, (0, 0), {(0, 2): 1.0}, slip=0.15, max_steps=8))
    rungs.append(make_grid_env("D3 4x4 gridworld", 4, 4, (3, 0), {(0, 3): 1.0}, walls={(1, 1), (2, 1)}, max_steps=28))
    rungs.append(make_grid_env("D4 stochastic windy grid", 5, 5, (4, 0), {(0, 4): 1.0}, walls={(1, 1), (2, 1), (3, 3)}, slip=0.10, wind={(3, 2): np.array([-1, 0]), (2, 3): np.array([-1, 0])}, max_steps=40))
    goals = {(0, 5): 1.0}
    if goal_conditioned:
        goals[(5, 0)] = 0.8
        goals[(2, 5)] = 0.7
    rungs.append(make_grid_env("D5 larger sparse-reward grid", 6, 6, (5, 0), goals, walls={(1, 1), (1, 2), (2, 2), (3, 4), (4, 1)}, slip=0.12, wind={(4, 3): np.array([-1, 0]), (3, 3): np.array([-1, 0])}, max_steps=55))
    return rungs


def plot_policy(ax, env, q_values, title):
    grid = np.full((env["height"], env["width"]), np.nan)
    arrows = ["↑", "→", "↓", "←"]
    for pos in env["states"]:
        state = env["index"][pos]
        grid[pos] = np.max(q_values[state])
    ax.imshow(grid, cmap="viridis")
    for pos in env["states"]:
        state = env["index"][pos]
        label = "G" if pos in env["goals"] else arrows[int(np.argmax(q_values[state]))]
        ax.text(pos[1], pos[0], label, ha="center", va="center", color="white")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

The lesson's common discounted-consequence calculation is the guardrail: rewards `[1, 0, 2]` with $\gamma=0.9$ give $G=2.620$, and the one-step target $1+0.9\cdot0.8$ gives $1.720$. This notebook then specializes that backup for Imitation & inverse RL.

$$\hat\pi(a\mid s)=\frac{N(s,a)+\lambda}{\sum_b N(s,b)+\lambda |A|}$$

In [ ]:

rewards = [1.0, 0.0, 2.0]
gamma = 0.9
three_step_return = discounted_return(rewards, gamma)
target, q_new = td_update(q_old=0.4, reward=1.0, next_value=0.8, alpha=0.5, gamma=gamma)

print("three-step return", round(three_step_return, 3))
print("one-step target", round(target, 3))
print("updated Q", round(q_new, 3))

assert round(three_step_return, 3) == 2.620
assert round(target, 3) == 1.720
assert round(q_new, 3) == 1.060


Now define `behavior_clone_and_reward_fit` once on D1. The assert is intentionally small and exact, so the same method can scale to D2-D5 without changing the objective or metric.

In [ ]:

def behavior_clone_and_reward_fit(demos, n_states, n_actions, smoothing=1.0):
    counts = np.full((n_states, n_actions), smoothing)
    rewards = np.zeros(n_states)
    visits = np.zeros(n_states)
    for state, action, reward, next_state in demos:
        counts[state, action] += 1.0
        rewards[next_state] += reward
        visits[next_state] += 1.0
    policy = counts / counts.sum(axis=1, keepdims=True)
    inferred_reward = rewards / np.maximum(visits, 1.0)
    return policy, inferred_reward


demos = []
for step in range(20):
    demos.append((0, 1, 1.0, 1))
policy, inferred_reward = behavior_clone_and_reward_fit(demos, n_states=2, n_actions=4)
cloned_action = int(np.argmax(policy[0]))
print("cloned action", ACTION_NAMES[cloned_action])
print("inferred goal reward", inferred_reward[1])

assert len(demos) == 20
assert cloned_action == 1
assert round(float(inferred_reward[1]), 3) == 1.000


## The dataset ladder

Family F12 is built inline: D1 is inspectable, D2 adds stochasticity, D3 is a 4x4 grid, D4 adds wind/slip, and D5 is a larger sparse-reward grid. The preview reports sizes before any learning code is used.

In [ ]:

rungs = rl_ladder(goal_conditioned=False)
for name, env in [(env["name"], env) for env in rungs]:
    size = len(env["states"])
    goal_count = len(env["goals"])
    print(f"{name}: states={size}, actions={len(ACTIONS)}, goals={goal_count}, slip={env['slip']}")
    print("  start", env["start"], "sample states", env["states"][:4])


## Run the same method across D1-D5

The code below keeps one algorithmic idea fixed and reports the plan metric: return. Seeds are fixed and all runs are CPU-only.

In [ ]:

def expert_action(env, pos):
    goal = max(env["goals"], key=env["goals"].get)
    best_action = 0
    best_distance = 1e9
    for action in ACTIONS:
        next_pos = move_position(env, pos, int(action))
        distance = abs(next_pos[0] - goal[0]) + abs(next_pos[1] - goal[1])
        if distance < best_distance:
            best_distance = distance
            best_action = int(action)
    return best_action


def collect_demos(env, episodes=20, noise=0.05, seed=0):
    local_rng = np.random.default_rng(seed)
    demos = []
    for episode in range(episodes):
        pos = env["start"]
        for step in range(env["max_steps"]):
            state = env["index"][pos]
            action = expert_action(env, pos)
            if local_rng.random() < noise:
                action = int(local_rng.choice(ACTIONS))
            next_pos, reward, done = sample_step(env, pos, action, local_rng)
            demos.append((state, action, reward, env["index"][next_pos]))
            pos = next_pos
            if done:
                break
    return demos


def q_from_cloned_policy(env, policy):
    q_values = np.log(policy + 1e-6)
    return q_values


rungs = rl_ladder()
rows = []
artifacts = []
for rung_id, env in enumerate(rungs, start=1):
    demos = collect_demos(env, episodes=20 + 5 * rung_id, noise=0.08, seed=1000 + rung_id)
    policy, inferred_reward = behavior_clone_and_reward_fit(demos, len(env["states"]), len(ACTIONS))
    cloned_q = q_from_cloned_policy(env, policy)
    learned_return, success = rollout_return(env, cloned_q, seed=1100 + rung_id)
    rows.append((env["name"], len(demos), learned_return, success))
    artifacts.append((env, cloned_q, np.array([learned_return])))

print("rung | demo steps | cloned return | success")
for name, demo_steps, learned_return, success in rows:
    print(f"{name} | {demo_steps} | {learned_return:.3f} | {success:.2f}")


## Results visualization

The closing figure has two parts: top-row small multiples show the learned artifact for each rung; the bottom-left summary curve plots the metric from D1 to D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, artifact in zip(axes[0], artifacts):
    env, q_values, history = artifact
    plot_policy(ax, env, q_values, env["name"].split()[0])
metric_values = []
for artifact in artifacts:
    history = artifact[2]
    metric_values.append(float(np.mean(history[-10:])))
axes[1, 0].plot(range(1, len(metric_values) + 1), metric_values, marker="o")
axes[1, 0].set_title("return vs rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("return")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5

Named pitfall: ignoring policy support causes compounding errors outside demonstrated states. The first line reproduces the wrong behavior; the fix keeps the lesson's discounted objective and improves the reported behavior.

In [ ]:

d5 = rl_ladder()[4]
narrow_demos = collect_demos(d5, episodes=5, noise=0.0, seed=2200)
wide_demos = collect_demos(d5, episodes=25, noise=0.15, seed=2300)
narrow_policy, narrow_reward = behavior_clone_and_reward_fit(narrow_demos, len(d5["states"]), len(ACTIONS), smoothing=0.01)
wide_policy, wide_reward = behavior_clone_and_reward_fit(wide_demos, len(d5["states"]), len(ACTIONS), smoothing=1.0)
narrow_q = q_from_cloned_policy(d5, narrow_policy)
wide_q = q_from_cloned_policy(d5, wide_policy)
narrow_return, narrow_success = rollout_return(d5, narrow_q, seed=2400)
wide_return, wide_success = rollout_return(d5, wide_q, seed=2500)
print("narrow-demo success", narrow_success)
print("noise-augmented DAgger-style success", wide_success)


## Evaluate it + Practice

- Metric: track return against a no-skill random or unsupported-action baseline.
- Sanity check: D1 must match the exact asserts above before trusting D5.
- Ablation: turn off the key idea (`behavior_clone_and_reward_fit`) and the metric should drop or become less stable.
- Failure signal: values improve while realized return, regret, or success rate does not.
- CPU rule: keep seeds fixed and do not scale episodes until the small ladder behaves.

Practice prompts:
1. Change $\gamma$ from 0.9 to 0.7 and predict which rung changes most.
2. Add one wall to D3 and rerun the same method without changing its API.
3. Replace the no-skill baseline with a hand-coded greedy baseline and compare.